# FLATCLASS · Part 1 — Allometric Baseline Models

**Series:** *Beyond the Scale: Can Synthetic Data Solve the Small-Dataset Problem in Aquaculture AI?*  
**Notebook:** `part1_allometric_baselines.ipynb`  
**Prerequisite:** `part1_eda_v2.ipynb` must have been run first (generates `real_train.parquet` and `real_test.parquet`)

---

### Purpose

Establish the **allometric baseline** on real training data only.  
This is the reference performance against which synthetic augmentation will be evaluated in Part 3.

Two models are fitted and compared:

| Model | Formula | Complexity |
|---|---|---|
| Univariate power law | W = a · L^b | 2 parameters |
| Multivariate power law | W = a · L^b₁ · A^b₂ · E^b₃ | 4 parameters |

### Design rules

- ⚠️ **Training set only.** The test set (`real_test.parquet`) is not touched in this notebook.
- All evaluation uses **5-fold cross-validation** on the training set.
- The Duan smearing correction is applied to all log-space predictions.
- Fitted models are saved to `../results/models/` for use in Part 3.
- Metrics are saved to `../results/metrics/baselines_cv.csv`.

### Dependencies
```
pandas · numpy · scipy · scikit-learn · matplotlib · seaborn · joblib
```


## 0. Environment Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import curve_fit

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import BaseEstimator, RegressorMixin
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

# ── Config ─────────────────────────────────────────────────────
RANDOM_STATE = 42
N_FOLDS      = 5
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", font_scale=1.1)
PALETTE  = {"uni": "#1f77b4", "multi": "#2ca02c", "ols": "#d62728"}
FIG_DIR  = Path("./results/figures")
MOD_DIR  = Path("./results/models")
MET_DIR  = Path("./results/metrics")
for d in [FIG_DIR, MOD_DIR, MET_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FEATURES = ["length_cm", "width_cm", "thickness_cm"]
TARGET   = "weight_g"
COL_LABELS = {
    "weight_g":    "Weight (g)",
    "length_cm":   "Length (cm)",
    "width_cm":    "Width (cm)",
    "thickness_cm":"Thickness (cm)",
}

print("Ready.")


## 1. Load Training Data

In [ ]:
df_train = pd.read_parquet("./data/processed/real_train.parquet")

# Keep only morphometric columns + target + traceability
cols = [TARGET] + FEATURES + ["data_origin", "bio_valid"]
cols = [c for c in cols if c in df_train.columns]
df_train = df_train[cols].copy()

print(f"Training set: {len(df_train)} records")
print(f"Columns     : {df_train.columns.tolist()}")
print(f"Flagged (bio_valid=False): {(~df_train['bio_valid']).sum() if 'bio_valid' in df_train.columns else 'N/A'}")
df_train.describe().round(3)


## 2. Model Definitions and Helpers

### Key design choice: Duan smearing correction

When we fit log(W) = f(log(L), log(A), log(E)) and back-transform:

    W_pred = exp(log_W_pred)

we introduce a systematic **negative bias** because E[exp(ε)] ≠ 1 for non-zero residuals ε.
The Duan (1983) smearing estimator corrects this:

    W_pred_corrected = smearing_factor × exp(log_W_pred)

where `smearing_factor = mean(exp(residuals_in_log_space))`, estimated from training fold residuals.

Reference: Duan, N. (1983). *Smearing estimate: a nonparametric retransformation method.*  
Journal of the American Statistical Association, 78(383), 605–610.


In [ ]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray,
                       label: str = "") -> dict:
    """Compute a standard set of regression metrics in original (gram) scale."""
    mae    = mean_absolute_error(y_true, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_true, y_pred))
    r2     = r2_score(y_true, y_pred)
    med_ae = np.median(np.abs(y_true - y_pred))
    bias   = np.mean(y_pred - y_true)
    mape   = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    if label:
        print(f"  {'Metric':<15} {'Value':>10}")
        print(f"  {'-'*27}")
        print(f"  {'MAE (g)':<15} {mae:>10.4f}")
        print(f"  {'RMSE (g)':<15} {rmse:>10.4f}")
        print(f"  {'Median AE (g)':<15} {med_ae:>10.4f}")
        print(f"  {'R²':<15} {r2:>10.4f}")
        print(f"  {'Bias (g)':<15} {bias:>10.4f}")
        print(f"  {'MAPE (%)':<15} {mape:>10.4f}")

    return {"model": label, "MAE": mae, "RMSE": rmse,
            "R2": r2, "Median_AE": med_ae, "Bias": bias, "MAPE": mape}


def error_by_quartile(y_true: np.ndarray, y_pred: np.ndarray,
                      label: str = "") -> pd.DataFrame:
    """Compute MAE by weight quartile to identify where models struggle."""
    df = pd.DataFrame({"observed": y_true, "predicted": y_pred})
    df["quartile"] = pd.qcut(y_true, q=4,
                             labels=["Q1\n(lightest)", "Q2", "Q3", "Q4\n(heaviest)"])
    df["abs_error"] = np.abs(y_true - y_pred)

    out = df.groupby("quartile", observed=True)["abs_error"].agg(
        MAE="mean", Median="median", N="count"
    ).round(4)
    if label:
        print(f"\nMAE by weight quartile — {label}:")
        display(out)
    return out


def power_law(L, a, b):
    """Univariate allometric power law: W = a * L^b"""
    return a * np.power(L, b)


def duan_smearing(y_true_log: np.ndarray, y_pred_log: np.ndarray) -> float:
    """
    Duan (1983) smearing factor for log-space retransformation bias correction.
    Returns the factor by which exp(log_predictions) should be multiplied.
    """
    return float(np.exp(y_true_log - y_pred_log).mean())


## 3. Model 1 — Univariate Power Law: W = a · L^b

We fit using two approaches:
1. **Nonlinear least squares** (`scipy.optimize.curve_fit`) — direct fit on original scale
2. **Log-linearisation** + Duan correction — standard allometric approach

Both should give nearly identical parameters. The log-linearisation approach  
integrates naturally into the cross-validation loop.


In [ ]:
X_all = df_train[FEATURES].values
y_all = df_train[TARGET].values
L_all = df_train["length_cm"].values

# ── Nonlinear fit (full training set, for parameter inspection) ────────────
popt, pcov = curve_fit(power_law, L_all, y_all, p0=[0.01, 3.0], maxfev=10000)
a_nls, b_nls = popt
perr = np.sqrt(np.diag(pcov))

print("Univariate W = a · L^b  (nonlinear least squares, full training set)")
print(f"  a = {a_nls:.6f}  ±  {perr[0]:.6f}")
print(f"  b = {b_nls:.4f}   ±  {perr[1]:.4f}")
print(f"  Biological note: b < 3 → negative allometry (expected for flatfish)")

# ── Log-linearisation (for CV loop) ───────────────────────────────────────
log_L = np.log(L_all)
log_y = np.log(y_all)

uni_loglin = LinearRegression().fit(log_L.reshape(-1, 1), log_y)
log_b = uni_loglin.coef_[0]
log_a_coef = uni_loglin.intercept_
smearing_uni_full = duan_smearing(log_y, uni_loglin.predict(log_L.reshape(-1, 1)))

print(f"\nLog-linearised: log(W) = {log_a_coef:.4f} + {log_b:.4f}·log(L)")
print(f"  a = exp({log_a_coef:.4f}) = {np.exp(log_a_coef):.6f}")
print(f"  b = {log_b:.4f}")
print(f"  Duan smearing factor: {smearing_uni_full:.4f}")


## 4. Model 2 — Multivariate Power Law: W = a · L^b₁ · A^b₂ · E^b₃

Log-linearised form:

    log(W) = log(a) + b₁·log(L) + b₂·log(A) + b₃·log(E)

This is a standard multiple linear regression in log space.


In [ ]:
log_X = np.log(df_train[FEATURES].values)
log_y = np.log(y_all)

multi_loglin = LinearRegression().fit(log_X, log_y)
b1, b2, b3   = multi_loglin.coef_
log_a_multi  = multi_loglin.intercept_
smearing_multi_full = duan_smearing(log_y, multi_loglin.predict(log_X))

print("Multivariate W = a · L^b₁ · A^b₂ · E^b₃  (log-linearised, full training set)")
print(f"  log(a) = {log_a_multi:.4f}   →   a = {np.exp(log_a_multi):.6f}")
print(f"  b₁ (Length)    = {b1:.4f}")
print(f"  b₂ (Width)     = {b2:.4f}")
print(f"  b₃ (Thickness) = {b3:.4f}")
print(f"  Duan smearing factor: {smearing_multi_full:.4f}")

print("\nBiological interpretation:")
print(f"  • b₁ ≈ {b1:.2f}: weight scales sub-cubically with length (flatfish geometry)")
print(f"  • b₂ ≈ {b2:.2f}: width contributes meaningfully — area dominates mass")
print(f"  • b₃ ≈ {b3:.2f}: thickness contributes least — dorsoventral axis constrained")
print(f"  • b₁+b₂+b₃ ≈ {b1+b2+b3:.2f} (isometric 3D expectation = 3.0)")


## 5. Cross-Validated Performance Comparison

We compare three models using 5-fold CV on the training set:
- **OLS baseline**: linear regression with all three features (raw scale)
- **Univariate allometric**: W = a·L^b (log-space fit + Duan)
- **Multivariate allometric**: W = a·L^b₁·A^b₂·E^b₃ (log-space fit + Duan)

> The test set is NOT used here. It remains sealed until Part 3.


In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results_cv = {
    "OLS_linear":    {"MAE": [], "RMSE": [], "R2": [], "Median_AE": []},
    "Univariate_W=aLb": {"MAE": [], "RMSE": [], "R2": [], "Median_AE": []},
    "Multivariate_W=aLbAcEd": {"MAE": [], "RMSE": [], "R2": [], "Median_AE": []},
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_all)):
    X_tr, X_val   = X_all[tr_idx], X_all[val_idx]
    y_tr, y_val   = y_all[tr_idx], y_all[val_idx]
    L_tr, L_val   = L_all[tr_idx], L_all[val_idx]

    # --- OLS linear baseline ---
    ols = LinearRegression().fit(X_tr, y_tr)
    y_hat_ols = ols.predict(X_val)
    for k, v in zip(["MAE","RMSE","R2","Median_AE"],
                    [mean_absolute_error(y_val, y_hat_ols),
                     np.sqrt(mean_squared_error(y_val, y_hat_ols)),
                     r2_score(y_val, y_hat_ols),
                     np.median(np.abs(y_val - y_hat_ols))]):
        results_cv["OLS_linear"][k].append(v)

    # --- Univariate allometric ---
    log_L_tr  = np.log(L_tr).reshape(-1, 1)
    log_y_tr  = np.log(y_tr)
    uni_m     = LinearRegression().fit(log_L_tr, log_y_tr)
    log_pred  = uni_m.predict(np.log(L_val).reshape(-1, 1))
    sf_uni    = duan_smearing(log_y_tr, uni_m.predict(log_L_tr))
    y_hat_uni = sf_uni * np.exp(log_pred)
    for k, v in zip(["MAE","RMSE","R2","Median_AE"],
                    [mean_absolute_error(y_val, y_hat_uni),
                     np.sqrt(mean_squared_error(y_val, y_hat_uni)),
                     r2_score(y_val, y_hat_uni),
                     np.median(np.abs(y_val - y_hat_uni))]):
        results_cv["Univariate_W=aLb"][k].append(v)

    # --- Multivariate allometric ---
    log_X_tr   = np.log(X_tr)
    log_y_tr   = np.log(y_tr)
    multi_m    = LinearRegression().fit(log_X_tr, log_y_tr)
    log_pred_m = multi_m.predict(np.log(X_val))
    sf_multi   = duan_smearing(log_y_tr, multi_m.predict(log_X_tr))
    y_hat_multi = sf_multi * np.exp(log_pred_m)
    for k, v in zip(["MAE","RMSE","R2","Median_AE"],
                    [mean_absolute_error(y_val, y_hat_multi),
                     np.sqrt(mean_squared_error(y_val, y_hat_multi)),
                     r2_score(y_val, y_hat_multi),
                     np.median(np.abs(y_val - y_hat_multi))]):
        results_cv["Multivariate_W=aLbAcEd"][k].append(v)

# ── Summary table ─────────────────────────────────────────────
rows = []
for model_name, fold_scores in results_cv.items():
    row = {"Model": model_name}
    for metric, values in fold_scores.items():
        row[f"{metric}_mean"] = np.mean(values)
        row[f"{metric}_std"]  = np.std(values)
    rows.append(row)

df_cv = pd.DataFrame(rows)
df_cv.to_csv(MET_DIR / "baselines_cv.csv", index=False)

print(f"{'Model':<30} {'MAE':>10} {'±':>6} {'RMSE':>10} {'±':>6} {'R²':>8} {'±':>6} {'MedAE':>8}")
print("-" * 90)
for _, r in df_cv.iterrows():
    print(f"{r['Model']:<30} "
          f"{r['MAE_mean']:>10.4f} {r['MAE_std']:>6.4f} "
          f"{r['RMSE_mean']:>10.4f} {r['RMSE_std']:>6.4f} "
          f"{r['R2_mean']:>8.4f} {r['R2_std']:>6.4f} "
          f"{r['Median_AE_mean']:>8.4f}")
print(f"\nResults saved → {MET_DIR / 'baselines_cv.csv'}")


## 6. Error by Weight Quartile

Where do the models struggle most? We compute MAE by weight quartile  
to identify systematic under- or over-estimation in specific size ranges.  
This matters for grading: errors in the upper quartile (heaviest fish) have the highest economic impact.


In [ ]:
# Refit on full training set for quartile analysis
log_L_all = np.log(L_all).reshape(-1, 1)
log_X_all = np.log(X_all)
log_y_all = np.log(y_all)

# OLS
ols_full    = LinearRegression().fit(X_all, y_all)
y_ols       = ols_full.predict(X_all)

# Univariate allometric
uni_full    = LinearRegression().fit(log_L_all, log_y_all)
sf_u        = duan_smearing(log_y_all, uni_full.predict(log_L_all))
y_uni       = sf_u * np.exp(uni_full.predict(log_L_all))

# Multivariate allometric
multi_full  = LinearRegression().fit(log_X_all, log_y_all)
sf_m        = duan_smearing(log_y_all, multi_full.predict(log_X_all))
y_multi     = sf_m * np.exp(multi_full.predict(log_X_all))

quartile_labels = ["Q1 (lightest)", "Q2", "Q3", "Q4 (heaviest)"]
quartile_col    = pd.qcut(y_all, q=4, labels=quartile_labels)

fig, ax = plt.subplots(figsize=(10, 5))
x_pos  = np.arange(4)
width  = 0.26

for i, (y_pred, label, color) in enumerate([
    (y_ols,   "OLS linear",       PALETTE["ols"]),
    (y_uni,   "Univariate W=aL^b",PALETTE["uni"]),
    (y_multi, "Multivariate",     PALETTE["multi"]),
]):
    mae_q = [mean_absolute_error(y_all[quartile_col == q], y_pred[quartile_col == q])
             for q in quartile_labels]
    ax.bar(x_pos + (i - 1) * width, mae_q, width,
           label=label, color=color, alpha=0.85, edgecolor="white")

ax.set_xticks(x_pos)
ax.set_xticklabels(quartile_labels, fontsize=10)
ax.set_ylabel("MAE (g)", fontsize=12)
ax.set_title("MAE by Weight Quartile — Training Set (in-sample)", fontsize=12)
ax.legend(fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / "baseline_mae_by_quartile.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")


## 7. Observed vs. Predicted — All Models

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Observed vs. Predicted — Training Set (in-sample)", fontsize=13)

for ax, (y_pred, label, color) in zip(axes, [
    (y_ols,   "OLS linear",        PALETTE["ols"]),
    (y_uni,   "Univariate W=aL^b", PALETTE["uni"]),
    (y_multi, "Multivariate",      PALETTE["multi"]),
]):
    lim = max(y_all.max(), y_pred.max()) * 1.05
    ax.scatter(y_pred, y_all, s=20, color=color, alpha=0.6,
               edgecolors="white", lw=0.3)
    ax.plot([0, lim], [0, lim], "k--", lw=1.3, alpha=0.5, label="1:1 line")
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel("Predicted (g)", fontsize=11)
    ax.set_ylabel("Observed (g)", fontsize=11)
    r2 = r2_score(y_all, y_pred)
    mae = mean_absolute_error(y_all, y_pred)
    ax.set_title(f"{label}\nR² = {r2:.3f}  |  MAE = {mae:.3f} g", fontsize=10)
    ax.legend(fontsize=9)
    ax.spines[["top","right"]].set_visible(False)
    ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "baseline_obs_vs_pred.png", dpi=150, bbox_inches="tight")
plt.show()


## 7b. Figure 2 — Three-Model Comparison (Medium article)

Generates `fig2_ols_vs_allometric.png` from **real training data**.  
This replaces the previously simulated version used as a placeholder.

Three panels — observed vs. predicted weight (g):
- **(A)** Linear OLS
- **(B)** Univariate W = a·L^b  
- **(C)** Multivariate W = a·L^b₁·A^b₂·E^b₃

All models fitted on the full training set (no CV here — in-sample fit for visual clarity).  
Colour encodes relative residual magnitude (green = low, red = high).

In [ ]:
import matplotlib.gridspec as gridspec

# ── Predictions from full-training-set fits ───────────────────────────────
# (uses ols_full, uni_full, multi_full + smearing factors from cells above)
y_ols_fig   = ols_full.predict(X_all)
y_uni_fig   = sf_u * np.exp(uni_full.predict(log_L_all))
y_multi_fig = sf_m * np.exp(multi_full.predict(log_X_all))

def rel_mag(residuals, fitted):
    mag = np.abs(residuals) / (fitted + 1e-6)
    return (mag - mag.min()) / (np.ptp(mag) + 1e-9)

c_ols   = rel_mag(y_all - y_ols_fig,   y_ols_fig)
c_uni   = rel_mag(y_all - y_uni_fig,   y_uni_fig)
c_multi = rel_mag(y_all - y_multi_fig, y_multi_fig)
cmap_fig = plt.cm.RdYlGn_r

# ── Layout ────────────────────────────────────────────────────────────────
fig2 = plt.figure(figsize=(17, 5.6))
gs   = gridspec.GridSpec(1, 3, wspace=0.38, left=0.06, right=0.88)
cax  = fig2.add_axes([0.90, 0.15, 0.018, 0.68])

def panel(ax, y_pred, y_true, colors, title):
    lim = max(y_true.max(), y_pred.max()) * 1.07
    ax.scatter(y_pred, y_true, c=colors, cmap=cmap_fig, s=22, alpha=0.75,
               edgecolors="none", vmin=0, vmax=1)
    for xi, yi, ci in zip(y_pred, y_true, colors):
        ax.plot([xi, xi], [xi, yi], color=cmap_fig(ci), lw=0.55, alpha=0.40)
    ax.plot([0, lim], [0, lim], "k--", lw=1.2, alpha=0.55)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel("Predicted weight (g)", fontsize=10)
    ax.set_ylabel("Observed weight (g)", fontsize=10)
    ax.set_title(title, fontsize=10, pad=8)
    ax.spines[["top","right"]].set_visible(False)
    return ax

# Metrics for titles
from sklearn.metrics import r2_score, mean_absolute_error
r2_ols  = r2_score(y_all, y_ols_fig)
r2_uni  = r2_score(y_all, y_uni_fig)
r2_mul  = r2_score(y_all, y_multi_fig)
mae_ols = mean_absolute_error(y_all, y_ols_fig)
mae_uni = mean_absolute_error(y_all, y_uni_fig)
mae_mul = mean_absolute_error(y_all, y_multi_fig)
b_uni_v  = float(uni_full.coef_[0])
b1v, b2v, b3v = multi_full.coef_

# Panel A — OLS
axA = panel(fig2.add_subplot(gs[0]), y_ols_fig, y_all, c_ols,
            f"(A)  Linear OLS  ·  W ~ L + A + E\nR² = {r2_ols:.3f}  |  MAE = {mae_ols:.3f} g")
axA.annotate("Large fish\nunderestimated",
             xy=(y_ols_fig.max()*0.75, y_all.max()*0.55),
             xytext=(y_ols_fig.max()*0.35, y_all.max()*0.80),
             fontsize=8, color="#c0392b",
             arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.1))

# Panel B — Univariate
axB = panel(fig2.add_subplot(gs[1]), y_uni_fig, y_all, c_uni,
            f"(B)  Univariate  ·  W = a·L^b\nb = {b_uni_v:.4f}  |  R² = {r2_uni:.3f}  |  MAE = {mae_uni:.3f} g")
lim_uni = max(y_all.max(), y_uni_fig.max()) * 1.07
axB.annotate(
    f"b = {b_uni_v:.2f} ≈ 3.0\nMild OVB:\nwidth signal absorbed",
    xy=(lim_uni * 0.70, lim_uni * 0.40),
    xytext=(lim_uni * 0.08, lim_uni * 0.62),
    fontsize=8, color="#d35400",
    arrowprops=dict(arrowstyle="->", color="#d35400", lw=1.1),
)

# Panel C — Multivariate
axC = panel(fig2.add_subplot(gs[2]), y_multi_fig, y_all, c_multi,
            f"(C)  Multivariate  ·  W = a·L^b₁·A^b₂·E^b₃\nb₁={b1v:.2f} · b₂={b2v:.2f} · b₃={b3v:.2f}  |  R² = {r2_mul:.3f}")
axC.annotate(
    f"a = {float(np.exp(multi_full.intercept_)):.4f}\n"
    f"b₁(L) = {b1v:.2f}\nb₂(A) = {b2v:.2f}\nb₃(E) = {b3v:.2f}\n"
    f"Σb = {b1v+b2v+b3v:.2f} < 3",
    xy=(0.03, 0.68), xycoords="axes fraction",
    fontsize=8, color="#1a5276",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#aed6f1", alpha=0.85)
)

# Shared colorbar
sm = plt.cm.ScalarMappable(cmap=cmap_fig, norm=plt.Normalize(0, 1))
sm.set_array([])
cb2 = fig2.colorbar(sm, cax=cax)
cb2.set_label("Relative residual magnitude\n(Low → High)", fontsize=9, labelpad=10)
cb2.set_ticks([0, 0.5, 1]); cb2.set_ticklabels(["Low", "Mid", "High"])

fig2.suptitle(
    "From heteroscedasticity to allometric precision — three models compared\n"
    f"(real S. senegalensis training data · n = {len(y_all)} observations)",
    fontsize=11, fontweight="bold", y=1.02)

out_fig2 = FIG_DIR / "fig2_ols_vs_allometric.png"
plt.savefig(out_fig2, dpi=160, bbox_inches="tight")
plt.show()
print(f"Figure 2 saved → {out_fig2}")
print(f"\nIn-sample metrics (training set):")
print(f"  OLS        R²={r2_ols:.3f}  MAE={mae_ols:.3f} g")
print(f"  Univariate R²={r2_uni:.3f}  MAE={mae_uni:.3f} g")
print(f"  Multivar.  R²={r2_mul:.3f}  MAE={mae_mul:.3f} g")


## 8. Save Models for Part 3

We save the fitted models and their smearing factors.  
Part 3 will load these to compute the true test-set performance.


In [ ]:
# Save sklearn models
joblib.dump(ols_full,   MOD_DIR / "ols_linear.pkl")
joblib.dump(uni_full,   MOD_DIR / "allometric_univariate.pkl")
joblib.dump(multi_full, MOD_DIR / "allometric_multivariate.pkl")

# Save smearing factors and model parameters
meta = {
    "univariate": {
        "duan_smearing": sf_u,
        "log_a": float(uni_full.intercept_),
        "b":     float(uni_full.coef_[0]),
        "a":     float(np.exp(uni_full.intercept_)),
    },
    "multivariate": {
        "duan_smearing": sf_m,
        "log_a": float(multi_full.intercept_),
        "b1_length":    float(multi_full.coef_[0]),
        "b2_width":     float(multi_full.coef_[1]),
        "b3_thickness": float(multi_full.coef_[2]),
        "a": float(np.exp(multi_full.intercept_)),
    },
}
with open(MOD_DIR / "allometric_params.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Models saved:")
for p in sorted(MOD_DIR.iterdir()):
    print(f"  {p.name}")

print("\nAllometric parameters:")
print(json.dumps(meta, indent=2))


## 9. Baseline Summary

| Model | CV MAE (g) | CV R² | Key insight |
|---|---|---|---|
| Linear OLS | highest | lowest | Heteroscedasticity penalises large fish |
| Univariate W = a·L^b | intermediate | intermediate | Power law + Duan correction recovers most of the gain |
| **Multivariate W = a·L^b₁·A^b₂·E^b₃** | **lowest** | **highest** | Width adds independent signal; thickness modest but real |

### Critical threshold for Part 3

> Any synthetic augmentation strategy evaluated in Part 3 must **outperform the multivariate  
> allometric baseline on the sealed test set** to be considered scientifically useful.  
> Distributional similarity scores (KSComplement, CorrelationSimilarity) are necessary  
> but not sufficient conditions for practical value.

---

**Next notebook → `part2_synthetic_generation.ipynb`**  
Generate synthetic data using GaussianCopula, CTGAN and TVAE.  
The train/test split and the baseline metrics established here remain fixed.
